# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is a binary classification task.

The model will classify each existing content item into one of two classes:

1 — needs refresh attention because impressions are declining

0 — does not currently show a clear decline

This is classification rather than scoring because the immediate business decision is whether an item should enter the refresh-review queue.

The prediction should be treated as decision support, not as an automatic decision to rewrite or delete content.

In [5]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target variable is needs_refresh.

It is created from the observed trend_direction column.

needs_refresh = 1 when trend_direction is down.

needs_refresh = 0 when trend_direction is stable, up, or flat.

The new category is excluded because new content does not yet have enough historical information to measure a reliable trend.

After removing new content, the dataset contains 27,764 usable rows.

Among these:

16,262 pages are labelled needs_refresh = 1.
11,502 pages are labelled needs_refresh = 0.

This is an observed proxy label because it is based on measured historical performance rather than a manually assigned label.

In [6]:
lane_df = df[
    df["trend_direction"].isin(
        ["down", "stable", "up", "flat"]
    )
].copy()

lane_df["needs_refresh"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

print("Rows:", len(lane_df))

print(lane_df["trend_direction"].value_counts())

print(lane_df["needs_refresh"].value_counts())

Rows: 27764
trend_direction
down      16262
stable     5962
up         4388
flat       1152
Name: count, dtype: int64
needs_refresh
1    16262
0    11502
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will evaluate the model using the F1-score.

The F1-score balances precision and recall, making it suitable because both kinds of mistakes have a cost.

A false positive means reviewing content that does not actually need refreshing.

A false negative means missing content whose performance is declining.

A model with an F1-score of approximately 0.65 or higher would be a reasonable starting point for decision support.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content page.

Each row represents one unique content_id.

Each row contains information about the page, including:

search demand

keyword competition

content type

search intent

word count

content age

update history

These characteristics are used to predict whether the page may require refreshing.

In [8]:
feature_columns = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "provider_used",
    "model_used",
    "content_age_days",
    "days_since_last_update"
]

ml_lane = lane_df[
    feature_columns + ["needs_refresh"]
]

print(ml_lane.shape)

ml_lane.head()

(27764, 13)


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,content_age_days,days_since_last_update,needs_refresh
0,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,187,20,1
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,445,25,1
2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,141,20,1
3,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,463,22,0
4,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,263,14,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule such as "refresh every article older than 180 days" would ignore many other factors that influence content performance.

Whether content declines depends on combinations of:

search volume

keyword competition

search intent

content type

article length

content age

time since the last update

These factors interact in complex ways that cannot be captured by one simple if-statement.

A machine learning model can learn these relationships and provide better decision support for identifying content that may need refreshing.

In [9]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print(df.dtypes)

print(df.describe(include="all").T)

Rows: 30000
Columns: 44
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days        

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.